In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss 
import numpy as np
import pickle

c:\Users\ManikanteshwaraReddy\AppData\Local\Programs\Python\Python311\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [ ]:
# Load the dataset
data = pd.read_csv('insurancedata.csv', dtype={
        'age': 'int32', 'gender': 'category', 'bmi': 'float32', 'children': 'int32',
        'smoker': 'category', 'medical_history': 'category', 'family_medical_history': 'category',
        'exercise_frequency': 'category', 'occupation': 'category', 'charges': 'float32'})


In [ ]:
# dataa

In [ ]:
# Combine relevant text fields into a single document
data['document'] = data.apply(lambda row: f"Age: {row['age']}, Gender: {row['gender']}, BMI: {row['bmi']}, Children: {row['children']}, \
                                        Smoker: {row['smoker']}, Medical History: {row['medical_history']}, \
                                        Family Medical History: {row['family_medical_history']}, \
                                        Exercise Frequency: {row['exercise_frequency']}, Occupation: {row['occupation']}, \
                                        Charges: {row['charges']}", axis=1)

In [2]:
# Load pre-trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')



c:\Users\ManikanteshwaraReddy\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
# Load the previously generated embeddings and data
with open("my_faiss1.pkl", 'rb') as f:
    data, embeddings = pickle.load(f)

# Normalize the embeddings
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

# Create FAISS index
d = embeddings.shape[1]  # Dimension of the embeddings
index = faiss.IndexFlatL2(d) 

# Add embeddings to the index
index.add(embeddings)

# Save the FAISS index
faiss.write_index(index, "my_faiss1.index")

# Serialize the index
pkl = faiss.serialize_index(index)

# Save the serialized index
with open('faiss1.index', 'wb') as f:
    f.write(pkl)


In [4]:
def retrieve(query, top_k=5):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    return data.iloc[indices[0]]

In [6]:
# Example usage
query = "Age: 30, Gender: female, BMI: 22, Children: 2, Smoker: no, Medical History: none, Family Medical History: none, Exercise Frequency: often, Occupation: engineer, Charges: 2000"
retrieved_docs = retrieve(query)
print(retrieved_docs)

        age  gender    bmi  children smoker medical_history  \
545971   26    male  34.00         1     no   Heart disease   
345584   30    male  22.00         4     no   Heart disease   
401478   60    male  38.00         3     no   Heart disease   
474724   22  female  30.25         1     no   Heart disease   
259684   41  female  47.50         3     no   Heart disease   

       family_medical_history exercise_frequency  occupation       charges  \
545971          Heart disease         Frequently     Student  20032.125000   
345584               Diabetes              Never     Student  19289.607422   
401478               Diabetes              Never  Unemployed  16419.187500   
474724          Heart disease       Occasionally     Student  17821.078125   
259684          Heart disease              Never     Student  19190.242188   

                                                 document  
545971  Age: 26, Gender: male, BMI: 34.0, Children: 1,...  
345584  Age: 30, Gender: male, B

In [5]:
from langchain_community.llms import Ollama

In [6]:
# Generate a response based on the retrieved documents and user query using an LLM
def generate_response(combined_docs, query):
    prompt_template = f"""
    Act as a chat system for Health Insurance domain take as which input health insurance parameters and output Quotes for Insurance.
    Based on the sample insurance data provide a quotation for health insurance based on input health condition of the user.need not consider region.

    ID, age, gender, bmi, children, smoker, region, medical_history, family_medical_history, exercise_frequency, occupation, coverage_level, charges:
    sample health insurance data: {combined_docs}

    ---

    Provide a Quotation in Rupees for the input health condition of the user based on the above sample health insurance data: 
    input health condition of the user: {query}
    """
    llm = Ollama(model="mistral:7b-instruct  ", temperature=0.3)
    response = llm.invoke(prompt_template)
    
    # Remove the unwanted text from the response
    unwanted_text = "Please note that this is an estimated quote only. The actual premium may vary based on the specific health insurance provider, plan details, location, and other factors. It's always best to consult with a licensed insurance agent or broker for accurate quotes tailored to your unique situation."
    response = response.replace(unwanted_text, "")
    
    return response 

In [9]:
# # user input 
# def get_user_input():
#     age = input("Enter age: ")
#     gender = input("Enter gender (male/female): ")
#     bmi = float(input("Enter your BMI:"))
#     children = input("Enter number of children: ")
#     smoker = input("Are you a smoker? (yes/no): ")
#     medical_history = input("Enter medical history: ")
#     family_medical_history = input("Enter family medical history: ")
#     exercise_frequency = input("Enter exercise frequency: ")

#     query = f"Age: {age}, Gender: {gender}, BMI: {bmi}, Children: {children}, Smoker: {smoker}, Medical History: {medical_history}, Family Medical History: {family_medical_history}, Exercise Frequency: {exercise_frequency}"
#     return query

In [7]:
def get_user_input():
    questions = [
        "Enter age: ",
        "Enter gender (male/female): ",
        "Enter your BMI: ",
        "Enter number of children: ",
        "Are you a smoker? (yes/no): ",
        "Enter medical history: ",
        "Enter family medical history: ",
        "Enter exercise frequency: "
    ]

    answers = []
    for question in questions:
        answer = input(question)
        answers.append(answer)

    age, gender, bmi, children, smoker, medical_history, family_medical_history, exercise_frequency = answers

    query = f"Age: {age}, Gender: {gender}, BMI: {bmi}, Children: {children}, Smoker: {smoker}, Medical History: {medical_history}, Family Medical History: {family_medical_history}, Exercise Frequency: {exercise_frequency}"
    return query


In [8]:
# Main
query = get_user_input()
retrieved_docs = retrieve(query)  # Assuming retrieve is defined elsewhere
combined_docs = "\n".join(retrieved_docs)
response = generate_response(combined_docs, query)
print(response)


 Based on the provided health insurance data and the input health condition of the user, I will provide an estimated quote for a health insurance policy. Please note that this is a hypothetical quotation and may not reflect actual prices offered by insurance companies.

Here's the breakdown of the factors affecting the premium:
1. Age: 45 years old (mid-aged)
2. Gender: Male
3. BMI: 31.5 (Overweight)
4. Children: 2
5. Smoker: No
6. Medical History: None
7. Family Medical History: Heart disease
8. Exercise Frequency: Rarely
9. Occupation: Not provided (assumed to be a sedentary job)
10. Coverage Level: Not provided (assumed to be standard coverage)

With these factors in mind, let's estimate the monthly premium for a health insurance policy:

- Mid-aged individuals typically have higher premiums compared to younger people due to increased risk of health issues.
- Overweight individuals may face higher premiums because they are at a greater risk of developing chronic conditions like diab